## Создание нейросети в Scikit learn для расчетов стационарной задачи переноса бинарного электролита в сечении канала

0. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn import metrics

1. Подготовка обучающих данных

In [ ]:
# считываем мой датасет и на всякий случай удаляем пустые строки
df = pd.read_csv('dataset.csv', sep=';', decimal=',').dropna()

# X - входной параметр (в моем случае потенциал ddfi: меняется от 0,1 до 3)
X = df[['ddfi']]
# y - то, что нейросеть будет предсказывать
targets = ['Y min', 'X min', 'Y max', 'X max']
y = df[targets]

# используем MinMaxScaler, чтобы привести все данные к диапазону [0, 1]
# это важно так как Xmin Xmax имеют масштаб сильно меньше чем Ymin Ymax
# если использовтаь StandardScaler то мы получим скачки на иксах
# так как он не так жестко уравнивает разброс данных как MinMaxScaler

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# делим на выборки: обучающая(80%) и тестовая(20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42 )

2. Создание нейросети, подбор гиперпараметров с помощью GridSearchCV или RandomizedSearchCV, обучение нейросети

In [ ]:
# mlp = MLPRegressor(max_iter=10000, random_state=42)
# param_grid = {
# 'hidden_layer_sizes': [(100, 100), (150,), (50, 50, 50)],
# 'activation': ['tanh', 'relu', 'logistic'],
# 'learning_rate_init': [0.001, 0.005]
# }
# gs = GridSearchCV(mlp, param_grid, cv=5, scoring='r2', n_jobs=-1)
# gs.fit(X_train, y_train)
# print(f"Лучшие параметры: {gs.best_params_}")

#Limited-memory Broyden–Fletcher–Goldfarb–Shanno
# tanh подходит для плавных физических кривых, чем relu

# сменила solver на 'lbfgs'. он намного лучше для точной подгонки под физические кривые
mlp = MLPRegressor(max_iter=10000, random_state=42, solver='lbfgs')
# сетка параметров для поиска лучшей архитектуры (Grid Search)
param_grid = {
    'hidden_layer_sizes': [(200, 200), (100, 100, 100),],
    'activation': ['relu'],
    # Для lbfgs learning_rate_init не так критичен, но alpha оставим маленьким
    'alpha': [1e-5, 1e-4]
}
 # GridSearchCV делает кросс-валидацию (cv=5) для каждой комбинации параметров
gs = GridSearchCV(mlp, param_grid, cv=5, scoring='r2', n_jobs=-1)
gs.fit(X_train, y_train)

print(f"Лучшие параметры: {gs.best_params_}")

3. Визуализация результатов и подсчет метрик качества

In [ ]:
# Получаем предсказания (они будут в масштабе [0, 1])
y_pred_scaled = gs.predict(X_test)
all_preds_scaled = gs.predict(X_scaled)

# обратное масштабирование
y_test_real = scaler_y.inverse_transform(y_test)
y_pred_real = scaler_y.inverse_transform(y_pred_scaled)
all_preds_real = scaler_y.inverse_transform(all_preds_scaled)

# считаем метрики по реальным значениям
print(f"MAE (Средняя абсолютная ошибка): {metrics.mean_absolute_error(y_test_real, y_pred_real):.6f}")
print(f"R2 Score (Коэффициент детерминации): {metrics.r2_score(y_test_real, y_pred_real):.4f}")

# рисуем графики
plt.style.use('seaborn-v0_8')
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, col_name in enumerate(targets):
    # точки из файла
    axes[i].scatter(df['ddfi'], df[col_name], color='green', s=15, alpha=0.6, label='Реальность')

    # предсказание нейросети (линия)
    # Используем данные, возвращенные к реальному масштабу
    axes[i].plot(df['ddfi'], all_preds_real[:, i], color='red', linewidth=2.5, label='Нейросеть')

    axes[i].set_title(f'Параметр: {col_name}')
    axes[i].set_xlabel('ddfi')
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()

plt.tight_layout()
plt.show()

Прогнозирование значений Ymin, Xmin, Ymax, Xmax в необходимом диапазоне

In [ ]:
start_val = 1.375
stop_val = 1.4
step_val = 0.0005
test_points = np.arange(start_val, stop_val + step_val, step_val)
print(f"Прогноз для диапазона {start_val} - {stop_val} с шагом {step_val}")
print("-" * 60)
results_list = []

for point in test_points:
    point = round(point, 8)

    df_point = pd.DataFrame([[point]], columns=['ddfi'])
    scaled_p = scaler_X.transform(df_point)
    pred_scaled = gs.predict(scaled_p)
    pred_real = scaler_y.inverse_transform(pred_scaled)

    res_line = f"{str(point).replace('.', ',')}; " + "; ".join([f"{val:.8f}".replace('.', ',') for val in pred_real[0]])

    print(res_line)
    results_list.append(res_line)